In [ ]:
# !pip install -U transformers

In [ ]:
from torch.utils.data import DataLoader , Dataset
from torch.amp import autocast, GradScaler

import torch
import torch.nn as nn


# import tqdm
# from tqdm import tqdm


import pandas as pd
import numpy as np
import re
import unicodedata
import os
import json

from sklearn.model_selection import train_test_split

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_curve, auc, precision_recall_curve
)

from sklearn.metrics import (
    accuracy_score, f1_score, precision_score,
    recall_score, roc_auc_score,
    matthews_corrcoef , log_loss, confusion_matrix
)


import joblib


from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer



from transformers import XLMRobertaTokenizer
from transformers import get_linear_schedule_with_warmup



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


In [ ]:
# ai_df = pd.read_json("/kaggle/input/nepali-ai-human/1_ai_dataset.jsonl",lines = True)

train_df = pd.read_json("/kaggle/input/datasets/ijita99/major-dataset/train.jsonl",lines = True)
test_df = pd.read_json("/kaggle/input/datasets/ijita99/major-dataset/test.jsonl",lines = True)
# valid_df = pd.read_json("/kaggle/input/datasets/sandeshpokharel999/nepali-dataset/valid.jsonl",lines = True)


# ML_MODEL_DIR = "/kaggle/input/datasets/sandespokharel/ai-detection-ml/outputs/models"   
# ai_df.head(10)

In [ ]:

stopwords = ['अगाडि','अझै','अनुसार','अन्य','अब','अरू','अरूलाई','अर्को','अलग','आए','आजको','आठ','आत्म','आदि','आफू',
'आफूलाई','आफैलाई','आफ्नै','आफ्नो','आयो','उदाहरण','उन','उनको','उनले','उप','उहाँलाई','एउटै','एक','एकदम','औं','कतै','कसरी','कसै',
'कसैले','कहाँबाट','कहिलेकाहीं','कहिल्यै','कहीं','का','कि','किन','किनभने','कुनै','कुरा','कृपया','के','केहि','केही','को','कोही','क्रमशः','गए','गरि','गरी','गरेका','गरेको','गरेर','गरौं','गर्छ','गर्छु',
'गर्दै','गर्न','गर्नु','गर्नुपर्छ','गर्ने','गर्यौं','गैर','चाँडै','चार','चाले','चाहनुहुन्छ','चाहन्छु','चाहिए','छ','छन्','छु','छैन','छौँ','छौं','जताततै',
'जब','जबकि','जसको','जसबाट','जसमा','जसलाई','जसले','जस्तै','जस्तो','जस्तोसुकै','जहाँ','जान',
'जाहिर','जुन','जे','जो','ठीक','त','तपाइँको','तपाईं','तर','तल','तापनि','तिनी','तथा','तिनीहरू','तिनीहरूको','तिनीहरूलाई',
'तिनीहरूले','तिमी','तिर','ती','तीन','तुरुन्तै','तेस्रो','त्यसपछि','त्यसमा','त्यसैले','त्यहाँ','त्यो','थिए','थिएन','थिएनन्','थियो','दिए','दिनुभएको','दिनुहुन्छ','दुई','देख','देखि','देखिन्छ','देखियो','देखे','देखेको',
'देखेर','देख्न','दोश्रो','दोस्रो','धेरै','न','नजिकै','नत्र','नयाँ','नि','नै','नौ','पक्का','पक्कै','पछि','पछिल्लो','पटक','पनि','पर्छ','पर्थ्यो','पहिले','पहिलो','पहिल्यै','पाँच','पाँचौं','पूर्व','प्रति',
'प्रत्येक','प्लस','फेरि','बने','बन्द','बन्न','बरु','बाटो','बारे','बाहिर','बाहेक','बीच','बीचमा','भए','भएको','भन','भने','भने्','भन्छन्','भन्छु','भन्दा','भन्नुभयो','भन्ने','भर','भित्र','भित्री','म','मलाई','मा','मात्र','माथि','मुख्य','मेरो',
'यति','यदि','यद्यपि','यस','यसको','यसपछि','यसबाहेक','यसरी','यसो','यस्तो','यहाँ','यहाँसम्म','या','यी','यो','र','रही','रहेका','रहेको','राखे','राख्छ','राम्रो','रूप','लगभग','लाई','लागि','ले','वरिपरि','वास्तवमा','वाहेक',
'विरुद्ध','विशेष','शायद','सँग','सँगै','सक्छ','सट्टा','सधैं','सबै','सबैलाई','समय','सम्भव','सम्म','सही','साँच्चै','सात','साथ','साथै','सायद','सारा','सो','सोध्न','सोही','स्पष्ट','हरे','हरेक','हामी',
'हामीलाई','हाम्रो','हुँ','हुन','हुने','हुनेछ','हुन्','हुन्छ','हो','होइन','होइनन्','होला','होस्']


# stopwords = []



nepali_cities = [
    "काठमाडौँ",
    "ललितपुर",
    "भक्तपुर",
    "पोखरा",
    "भरतपुर",
    "बुटवल",
    "भैरहवा",
    "नेपालगञ्ज",
    "धरान",
    "इटहरी",
    "बिराटनगर",
    "बिर्तामोड",
    "जनकपुर",
    "वीरगञ्ज",
    "हेटौंडा",
    "दमक",
    "धनगढी",
    "दिपायल",
    "दाङ",
    "तुलसीपुर",
    "घोराही",
    "गोरखा",
    "बागलुङ",
    "बेनी",
    "म्याग्दी",
    "तन्सेन",
    "रामेछाप",
    "धुलिखेल",
    "बनेपा",
    "पनौती",
    "चरिकोट",
    "भद्रपुर",
    "इलाम",
    "खाँदबारी",
    "इनरुवा",
    "राजविराज",
    "लहान",
    "सिरहा",
    "गौर",
    "कलैया",
    "मलंगवा",
    "जलेश्वर",
    "गुल्मी",
    "अर्घाखाँची",
    "प्युठान",
    "मुसिकोट",
    "रोल्पा",
    "रुकुम",
    "जाजरकोट",
    "सुर्खेत",
    "दैलेख",
    "वीरेन्द्रनगर",
    "बाजुरा",
    "अछाम",
    "डोटी",
    "बैतडी",
    "दार्चुला",
    "महेन्द्रनगर",
    "टीकापुर",
    "लम्की",
    "राजापुर",
    "पोखरा"
]


time_context_words = [
    "आज",
    "आजकल",
    "भोलि",
    "हिजो",
    # "अहिले",
    # "हाल",
    # "हालै",
    # "सधैं",
    # "कहिले",
    # "समय"
]

country_words = [
    "नेपाल",
    "नेपाली",
    # "नेपालको",
    # "नेपालमा",
    # "नेपालबाट",
    # "नेपाललाई",
    # "नेपालकै",
    # "देश",
    # "देशको",
    # "देशमा",
    # "देशलाई",
    # "देशबाट",
    # "राष्ट्र",
    # "राष्ट्रिय",
    # "राष्ट्रको",
    # "राष्ट्रमा",
    # "राज्य",
    # "सरकार",
    # "सरकारी"
]

common_noise_words = [
    "मान्छे",
    "मानिस",
    "जीवन",
    "काम",
    "कुरा",
    "समस्या",
    "समाधान",
    "विषय",
    "अवस्था",
    "कारण",
    "परिणाम",
    "स्थिति",
    "प्रक्रिया",
    "तथ्य",
    "विचार"
]

identity_words = [
    "भाषा",
    "नेपालीभाषा",
    "नागरिक",
    "नागरिकता",
    "नागरिकको",
    "जनता",
    "जनताको",
    "समाज",
    "समाजको",
    "समाजमा"
]



stopwords_set = set(stopwords)


# stopwords_set.update(nepali_cities)
# stopwords_set.update(time_context_words)
# stopwords_set.update(country_words)

In [ ]:
# human_df = pd.read_json("/kaggle/input/nepali-ai-human/1_human_dataset.jsonl",lines = True)


# human_df.head()

In [ ]:
test_df = test_df.sample(frac=1, random_state=42).reset_index(drop=True)
train_df = train_df.sample(frac=1, random_state=42).reset_index(drop=True)
# valid_df = valid_df.sample(frac=1, random_state=42).reset_index(drop=True)

print('-------------------loded dataset-------------------')

In [ ]:
def clean_nepali_text(text):
    if pd.isna(text):
        return ""

    # 1. Normalize Unicode (Crucial for Nepali)
    text = unicodedata.normalize("NFKC", text)

    # 2. Remove URLs and HTML first (before symbols are removed)
    text = re.sub(r"https?://\S+|www\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)

    # 3. Remove English letters and Numbers
    text = re.sub(r"[A-Za-z0-9०-९]", " ", text)

    # 4. Remove Punctuation and Special Symbols
    # This keeps only Nepali characters and spaces
    text = re.sub(r'[।,!?]', ' ', text)


    text = re.sub(r"[^\u0900-\u097F\s]", "", text)

    # 5. Remove extra spaces
    text = re.sub(r"\s+", " ", text).strip()

    # 6. Smart Stopword Removal
    words = text.split()
    cleaned_words = []
    
    for w in words:
        # Check for exact match in stopwords
        if w in stopwords_set:
            continue
            
        # Optional: Remove common suffixes if the word ends with them
        # Note: This is a basic stemmer logic. 
        # for suffix in ['को', 'ले', 'मा', 'लाई', 'भन्दा']:
        #     if w.endswith(suffix) and len(w) > len(suffix):
        #         w = w[:-len(suffix)]
        
        if len(w) > 1:
            cleaned_words.append(w)

    return " ".join(cleaned_words)


# combined_df = combined_text_df

# combined_df["text"] = combined_df["text"].apply(clean_nepali_text)
# combined_df.to_json("nepali_dataset.jsonl", orient="records", lines=True, force_ascii=False)


In [ ]:
mapping = {"human":0,"ai":1}
 

train_df['label'] = train_df['label'].map(mapping)
test_df['label'] =  test_df['label'].map(mapping)
# valid_df['label'] =  valid_df['label'].map(mapping)



train_df["text"] = train_df["text"].apply(clean_nepali_text)
test_df["text"] = test_df["text"].apply(clean_nepali_text)
# valid_df["text"] = valid_df["text"].apply(clean_nepali_text)


In [ ]:
# from wordcloud import WordCloud
# import matplotlib.pyplot as plt
# def generate_wordcloud(text, title):
#     wc = WordCloud(
#         font_path="/kaggle/input/nepali-font/NotoSansDevanagari-Regular.ttf",  # Devanagari font for Nepali
#         background_color="white",
#         width=800,
#         height=400,
#         regexp=r'[\u0900-\u097F]{3,}',  # match 2 or more Nepali letters as a word

#         collocations = False
#     ).generate(text)
    
#     plt.figure(figsize=(10, 5))
#     plt.imshow(wc, interpolation="bilinear")
#     plt.axis("off")
#     plt.title(title, fontsize=16)
#     plt.show()

# # ---------------------------
# # Create word clouds
# # ---------------------------
# ai_text = " ".join(combined_df.loc[combined_df["label"] == 1, "text"])
# human_text = " ".join(combined_df.loc[combined_df["label"] == 0, "text"])

# generate_wordcloud(ai_text, "AI-written Nepali Text")
# generate_wordcloud(human_text, "Human-generated Nepali Text")

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load multilingual MiniLM for sequence classification

model_name ="Multilingual-MiniLM-L12-H384"

model = AutoModelForSequenceClassification.from_pretrained(
    f'microsoft/{model_name}',
    num_labels=2, 
    problem_type="single_label_classification"
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/Multilingual-MiniLM-L12-H384")

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze last 3 transformer layers
for layer in model.bert.encoder.layer[-6:]:
    for param in layer.parameters():
        param.requires_grad = True

# Unfreeze pooler
for param in model.bert.pooler.parameters():
    param.requires_grad = True

# Unfreeze classifier head
for param in model.classifier.parameters():
    param.requires_grad = True

# Verify — print trainable vs frozen
total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
frozen_params    = total_params - trainable_params

print(f"Total      : {total_params:,}")
print(f"Trainable  : {trainable_params:,} ({100 * trainable_params / total_params:.1f}%)")
print(f"Frozen     : {frozen_params:,} ({100 * frozen_params / total_params:.1f}%)")




In [ ]:
# from transformers import XLMRobertaForSequenceClassification
# from transformers import XLMRobertaModel , XLMRobertaTokenizer

# model_name = "xlm-roberta-base"
# model = XLMRobertaForSequenceClassification.from_pretrained(
#     model_name,
#     num_labels=2, 
#     problem_type="single_label_classification")
# tokenizer = XLMRobertaTokenizer.from_pretrained('xlm-roberta-base')

# # Freeze everything first
# for param in model.parameters():
#     param.requires_grad = False

# # Unfreeze last 8 transformer layers
# for layer in model.roberta.encoder.layer[-6:]:
#     for param in layer.parameters():
#         param.requires_grad = True

# # Unfreeze classifier head
# for param in model.classifier.parameters():
#     param.requires_grad = True

# total_params     = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# frozen_params    = total_params - trainable_params

# print(f"Total      : {total_params:,}")
# print(f"Trainable  : {trainable_params:,} ({100 * trainable_params / total_params:.1f}%)")
# print(f"Frozen     : {frozen_params:,} ({100 * frozen_params / total_params:.1f}%)")

In [ ]:
# from transformers import AutoModelForSequenceClassification, AutoTokenizer

# # Define the NepBERTa model path from Hugging Face
# model_name = "nepberta"

# full_model_name = f'dura-garage/{model_name}'



# # Load the specialized tokenizer
# tokenizer = AutoTokenizer.from_pretrained(full_model_name)

# # Load the model for sequence classification
# model = AutoModelForSequenceClassification.from_pretrained(
#     full_model_name, 
#     num_labels=2, 
#     problem_type="single_label_classification",
#     # from_tf=True
# )


# # Freeze everything first
# for param in model.parameters():
#     param.requires_grad = False

# # Unfreeze last 3 transformer layers
# for layer in model.bert.encoder.layer[-8:]:
#     for param in layer.parameters():
#         param.requires_grad = True

# # Unfreeze pooler
# for param in model.bert.pooler.parameters():
#     param.requires_grad = True

# # Unfreeze classifier head
# for param in model.classifier.parameters():
#     param.requires_grad = True

# # Verify — print trainable vs frozen
# total_params     = sum(p.numel() for p in model.parameters())
# trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
# frozen_params    = total_params - trainable_params

# print(f"Total      : {total_params:,}")
# print(f"Trainable  : {trainable_params:,} ({100 * trainable_params / total_params:.1f}%)")
# print(f"Frozen     : {frozen_params:,} ({100 * frozen_params / total_params:.1f}%)")




In [ ]:
# model = torch.compile(model)

In [ ]:


class ChunkedTextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512, overlap=30, min_tokens=70):
        self.input_ids = []
        self.attention_mask = []
        self.labels = []
        
        print(f"Processing {len(df)} documents...")
        
        # We manually process each document because the slow tokenizer won't do it correctly
        for _, row in df.iterrows():
            text = str(row['text'])
            label = row['label']
            
            # 1. Encode full text (no truncation yet)
            full_ids = tokenizer.encode(text, add_special_tokens=False, truncation=False)
            
            # If the text is empty or too short to even bother with
            if len(full_ids) < min_tokens:
                continue

            # 2. Sliding Window logic
            # Window size minus special tokens ([CLS], [SEP])
            content_len = max_len - 2
            step = content_len - overlap
            
            for i in range(0, len(full_ids), step):
                chunk = full_ids[i : i + content_len]
                
                # Check if the last chunk meets the min_tokens requirement
                if len(chunk) < min_tokens:
                    break
                
                # Add special tokens and pad manually
                # This ensures every chunk is exactly max_len
                chunk_ids = [tokenizer.cls_token_id] + chunk + [tokenizer.sep_token_id]
                mask = [1] * len(chunk_ids)
                
                # Padding
                padding_len = max_len - len(chunk_ids)
                if padding_len > 0:
                    chunk_ids += [tokenizer.pad_token_id] * padding_len
                    mask += [0] * padding_len
                
                self.input_ids.append(chunk_ids)
                self.attention_mask.append(mask)
                self.labels.append(label)
                
                # Stop if we have reached the end of the text
                if i + content_len >= len(full_ids):
                    break

        # Convert to tensors
        self.input_ids = torch.tensor(self.input_ids, dtype=torch.long)
        self.attention_mask = torch.tensor(self.attention_mask, dtype=torch.long)
        # Ensure labels are Long tensors (usually required for CrossEntropyLoss)
        self.labels = torch.tensor(self.labels, dtype=torch.long)

        print(f"✅ Created {len(self.labels)} valid chunks from {len(df)} documents.")

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        # CHANGE: Return a tuple instead of a dict to match the unpacking 
        # in the training loop: "for input_ids, attention_mask, labels in dataloader"
        return self.input_ids[index], self.attention_mask[index], self.labels[index]

In [ ]:
class TextDataset(Dataset):
    def __init__(self, df, tokenizer, max_len):
        self.data = df
        self.tokenizer = tokenizer
        self.max_len = max_len
        self.labels = torch.tensor(df['label'].values, dtype=torch.long)


    def __len__(self):
        return len(self.data)

    def __getitem__(self, index):
        text = str(self.data.iloc[index]['text'])
        label = self.data.iloc[index]['label']

        encoded = self.tokenizer(
            text,
            max_length=self.max_len,
            truncation=True,
            padding="max_length",
            return_tensors="pt"
        )

        # .squeeze(0) removes the batch dim that return_tensors="pt" adds
        input_ids = encoded.input_ids.squeeze(0)           # (max_len,)
        attention_mask = encoded.attention_mask.squeeze(0) # (max_len,)
        label = torch.tensor(label, dtype=torch.long)

        return input_ids, attention_mask, label

In [ ]:
# train_df, test_df = train_test_split(combined_df, test_size=0.2, random_state=43)

# train_dataset = ChunkedTextDataset(train_df, tokenizer, max_len=512)
# test_dataset  = ChunkedTextDataset(test_df,  tokenizer, max_len=512)

# # train_dataset = TextDataset(train_df, tokenizer, max_len=512)
# # test_dataset  = TextDataset(test_df,  tokenizer, max_len=512)

# train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
# test_dataloader  = DataLoader(test_dataset,  batch_size=16)

In [ ]:

max_len = 512
batch_size = 16


# train_dataset = ChunkedTextDataset(train_df, tokenizer, max_len=max_len)
# test_dataset  = ChunkedTextDataset(test_df,  tokenizer, max_len=max_len)

train_dataset = TextDataset(train_df, tokenizer, max_len=max_len)
test_dataset  = TextDataset(test_df,  tokenizer, max_len=max_len)


train_dataloader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_dataloader  = DataLoader(test_dataset,  batch_size=batch_size)

# valid_dataloader  = DataLoader(valid_dataset,  batch_size=16)





# valid_dataset  = ChunkedTextDataset(valid_df,  tokenizer, max_len=512)
# valid_dataset  = ChunkedTextDataset(valid_df,  tokenizer, max_len=512)

In [ ]:
def print_split_stats(name, dataset):
    labels = dataset.labels
    n_human = (labels == 0).sum().item()
    n_ai    = (labels == 1).sum().item()
    print(f"\n📊 {name} Split")
    print(f"   Total chunks : {len(labels)}")
    print(f"   Human (0)    : {n_human}  ({100*n_human/len(labels):.1f}%)")
    print(f"   AI    (1)    : {n_ai}     ({100*n_ai/len(labels):.1f}%)")

print_split_stats("Train", train_dataset)
print_split_stats("Test",  test_dataset)
# print_split_stats("Valid",  valid_dataset)

In [ ]:


def get_metrics(all_labels, all_preds, all_probs=None):
    metrics = {
        "accuracy":  accuracy_score(all_labels, all_preds),
        "f1":        f1_score(all_labels, all_preds, average='weighted', zero_division=0),
        "precision": precision_score(all_labels, all_preds, average='weighted', zero_division=0),
        "recall":    recall_score(all_labels, all_preds, average='weighted', zero_division=0),
        "mcc":       matthews_corrcoef(all_labels, all_preds),  # reliable for imbalanced classes
    }
    if all_probs is not None:
        # AUC-ROC needs probability scores, not just predictions
        metrics["auc_roc"] = roc_auc_score(all_labels, all_probs[:, 1])
    return metrics



In [ ]:
scaler = GradScaler('cuda', enabled=device.type == 'cuda')

def train(model, dataloader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss = 0
    all_preds  = []
    all_labels = []
    all_probs  = []
    
    for input_ids, attention_mask, labels in dataloader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)
        optimizer.zero_grad(set_to_none=True)
        
        with autocast('cuda', enabled=device.type == 'cuda'):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss    = criterion(outputs.logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()
        
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(outputs.logits, dim=1)
        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.detach().cpu())
        all_probs.append(probs.detach().cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.cat(all_probs).numpy()
    metrics = get_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = total_loss / len(dataloader)
    return metrics

In [ ]:
eval_criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def evaluate(model, dataloader, device):
    model.eval()
    total_loss = 0
    all_preds  = []
    all_labels = []
    all_probs  = []
    with torch.inference_mode():
        for input_ids, attention_mask, labels in dataloader:
            input_ids      = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels         = labels.to(device)
            with autocast('cuda', enabled=device.type == 'cuda'):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
                loss    = eval_criterion(outputs.logits, labels)
            total_loss += loss.item()
            
            probs = torch.softmax(outputs.logits, dim=1)
            preds = torch.argmax(outputs.logits, dim=1)
            all_preds.append(preds.detach().cpu())
            all_labels.append(labels.detach().cpu())
            all_probs.append(probs.detach().cpu())

    all_preds  = torch.cat(all_preds).numpy()
    all_labels = torch.cat(all_labels).numpy()
    all_probs  = torch.cat(all_probs).numpy()
    metrics = get_metrics(all_labels, all_preds, all_probs)
    metrics["loss"] = total_loss / len(dataloader)
    return metrics



In [ ]:
model = model.to(device)

# lr = 1e-5
lr = 2e-5


optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=lr
)


epochs = 10



total_steps = len(train_dataloader) * epochs


scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=total_steps // 10,
    num_training_steps=total_steps
)


In [ ]:
# train_labels = train_dataset.labels
# n_human = (train_labels == 0).sum().item()
# n_ai    = (train_labels == 1).sum().item()
# total   = len(train_labels)

# class_weights = torch.tensor([
#     total / (2 * n_human),
#     total / (2 * n_ai)
# ], dtype=torch.float).to(device)



# criterion  = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)



In [ ]:
history = {
    "train": {"loss":[], "accuracy":[], "f1":[], "precision":[], "recall":[], "mcc":[], "auc_roc":[]},
    "test":  {"loss":[], "accuracy":[], "f1":[], "precision":[], "recall":[], "mcc":[], "auc_roc":[]}
}



# ── Best-model tracking ───────────────────────────────────────────────────────
CKPT_DIR       = "./saved_model"
METRICS_PATH   = os.path.join(CKPT_DIR, "best_metrics.json")
best_val_loss  = float("inf")
best_epoch     = -1
best_model = model
# ─────────────────────────────────────────────────────────────────────────────


In [ ]:
tokenizer.save_pretrained(CKPT_DIR)

for epoch in range(epochs):
    print(f"\nEpoch {epoch+1}/{epochs}")
    train_metrics = train(model, train_dataloader, optimizer, scheduler, criterion, device)
    test_metrics = evaluate(model, test_dataloader, device)
    
    print(f"Train — Loss: {train_metrics['loss']:.4f} | Acc: {train_metrics['accuracy']:.4f} | "
          f"F1: {train_metrics['f1']:.4f} | AUC: {train_metrics['auc_roc']:.4f} | MCC: {train_metrics['mcc']:.4f}")

    print(f"Test  — Loss: {test_metrics['loss']:.4f} | Acc: {test_metrics['accuracy']:.4f} | "
          f"F1: {test_metrics['f1']:.4f} | AUC: {test_metrics['auc_roc']:.4f} | MCC: {test_metrics['mcc']:.4f}")
    print("-" * 70)

    for k, v in train_metrics.items():
        history["train"][k].append(v)
    for k, v in test_metrics.items():
        history["test"][k].append(v)

    # ── Save only when val loss improves ─────────────────────────────────────
    if test_metrics["loss"] < best_val_loss:
        best_val_loss = test_metrics["loss"]
        best_epoch    = epoch + 1
        best_model = model

        model.save_pretrained(CKPT_DIR)

        best_metrics_snapshot = {
            "epoch":       best_epoch,
            "best_val_loss": best_val_loss,
            "train":       train_metrics,
            "test":        test_metrics,
        }
        with open(METRICS_PATH, "w") as f:
            json.dump(best_metrics_snapshot, f, indent=2)

        print(f"  ✓ New best model saved (val loss {best_val_loss:.4f})")
    # ─────────────────────────────────────────────────────────────────────────


model.save_pretrained('./final/')
tokenizer.save_pretrained('./final/')


print(f"\nTraining complete. Best checkpoint: epoch {best_epoch}, val loss {best_val_loss:.4f}")

In [ ]:
metrics_config = [
    ("loss",      "Loss",           "Loss over Epochs"),
    ("accuracy",  "Accuracy",       "Accuracy over Epochs"),
    ("f1",        "F1 Score",       "F1 over Epochs"),
    ("precision", "Precision",      "Precision over Epochs"),
    ("recall",    "Recall",         "Recall over Epochs"),
    ("mcc",       "MCC",            "Matthews Correlation Coefficient"),
    ("auc_roc",   "AUC-ROC",        "AUC-ROC over Epochs"),
]

epoch_range = range(1, len(history["train"]["loss"]) + 1)

fig, axes = plt.subplots(2, 4, figsize=(24, 10))
axes = axes.flatten()

for i, (key, ylabel, title) in enumerate(metrics_config):
    axes[i].plot(epoch_range, history["train"][key], label="Train", marker='o')
    axes[i].plot(epoch_range, history["test"][key],  label="Test",  marker='o')
    # axes[i].axvline(best_epoch, color='red', linestyle='--', linewidth=1.2, label=f'Best (ep {best_epoch})')
    axes[i].set_xlabel("Epoch")
    axes[i].set_ylabel(ylabel)
    axes[i].set_title(title)
    axes[i].legend()
    axes[i].grid(True)

# Hide the unused 8th subplot (2x4 grid = 8, we only have 7 metrics)
axes[-1].set_visible(False)

plt.tight_layout()
plt.savefig("training_metrics.png", dpi=300, bbox_inches="tight")
plt.show()
print("Training metrics plot saved as 'training_metrics.png'")


In [ ]:


best_model.eval()
all_preds  = []
all_labels = []
all_probs  = []

with torch.inference_mode():
    for input_ids, attention_mask, labels in test_dataloader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels         = labels.to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs   = torch.softmax(outputs.logits, dim=1)
        preds   = torch.argmax(probs, dim=1)

        all_preds.append(preds.detach().cpu())
        all_labels.append(labels.detach().cpu())
        all_probs.append(probs.detach().cpu())

all_preds  = torch.cat(all_preds).numpy()
all_labels = torch.cat(all_labels).numpy()
all_probs  = torch.cat(all_probs).numpy()

# ── Classification Report ────────────────────────────────────────────────────
print("\nClassification Report:\n")
print(classification_report(
    all_labels, all_preds,
    labels=[0, 1],
    target_names=['Human', 'AI'],
    zero_division=0
))

# ── Confusion Matrix & Rates ─────────────────────────────────────────────────
conf_matrix = confusion_matrix(all_labels, all_preds)
TN, FP, FN, TP = conf_matrix.ravel()

TPR = TP / (TP + FN + 1e-8)   # sensitivity / recall
TNR = TN / (TN + FP + 1e-8)   # specificity
FPR = FP / (FP + TN + 1e-8)   # fall-out
FNR = FN / (FN + TP + 1e-8)   # miss rate

print(f"True  Positive Rate (TPR/Sensitivity) : {TPR:.4f}")
print(f"True  Negative Rate (TNR/Specificity) : {TNR:.4f}")
print(f"False Positive Rate (FPR/Fall-out)    : {FPR:.4f}")
print(f"False Negative Rate (FNR/Miss Rate)   : {FNR:.4f}")

# ── ROC & PR curves ──────────────────────────────────────────────────────────
fpr_curve, tpr_curve, _ = roc_curve(all_labels, all_probs[:, 1])
roc_auc = auc(fpr_curve, tpr_curve)

precision_curve, recall_curve, _ = precision_recall_curve(all_labels, all_probs[:, 1])
pr_auc = auc(recall_curve, precision_curve)

# ── Plots ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(22, 16))
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.35)

CLASS_NAMES = ['Human', 'AI']

# 1. Raw confusion matrix
ax1 = fig.add_subplot(gs[0, 0])
sns.heatmap(
    conf_matrix, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax1
)
ax1.set_xlabel("Predicted")
ax1.set_ylabel("Actual")
ax1.set_title("Confusion Matrix (Counts)")

# 2. Normalised confusion matrix
ax2 = fig.add_subplot(gs[0, 1])
conf_norm = conf_matrix.astype(float) / conf_matrix.sum(axis=1, keepdims=True)
sns.heatmap(
    conf_norm, annot=True, fmt='.2%', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax2,
    vmin=0, vmax=1
)
ax2.set_xlabel("Predicted")
ax2.set_ylabel("Actual")
ax2.set_title("Confusion Matrix (Normalised)")

# 3. TPR / TNR / FPR / FNR bar chart
ax3 = fig.add_subplot(gs[0, 2])
rate_names  = ['TPR\n(Sensitivity)', 'TNR\n(Specificity)', 'FPR\n(Fall-out)', 'FNR\n(Miss Rate)']
rate_values = [TPR, TNR, FPR, FNR]
rate_colors = ['#2ecc71', '#3498db', '#e67e22', '#e74c3c']
bars = ax3.bar(rate_names, rate_values, color=rate_colors, edgecolor='white', linewidth=1.2)
for bar, val in zip(bars, rate_values):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.3f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
ax3.set_ylim(0, 1.12)
ax3.set_ylabel("Proportion")
ax3.set_title("Classification Rates")
ax3.grid(axis='y', alpha=0.4)

# 4. ROC curve
ax4 = fig.add_subplot(gs[1, 0])
ax4.plot(fpr_curve, tpr_curve, color='#3498db', lw=2, label=f'AUC = {roc_auc:.4f}')
ax4.plot([0, 1], [0, 1], color='grey', lw=1, linestyle='--', label='Random')
ax4.fill_between(fpr_curve, tpr_curve, alpha=0.1, color='#3498db')
ax4.set_xlabel("False Positive Rate")
ax4.set_ylabel("True Positive Rate")
ax4.set_title("ROC Curve")
ax4.legend(loc='lower right')
ax4.grid(alpha=0.3)

# 5. Precision-Recall curve
ax5 = fig.add_subplot(gs[1, 1])
ax5.plot(recall_curve, precision_curve, color='#2ecc71', lw=2, label=f'AUC = {pr_auc:.4f}')
ax5.fill_between(recall_curve, precision_curve, alpha=0.1, color='#2ecc71')
ax5.set_xlabel("Recall")
ax5.set_ylabel("Precision")
ax5.set_title("Precision-Recall Curve")
ax5.legend(loc='lower left')
ax5.grid(alpha=0.3)

# 6. Predicted probability distribution
ax6 = fig.add_subplot(gs[1, 2])
human_probs = all_probs[all_labels == 0, 1]   # P(AI) for actual Human samples
ai_probs    = all_probs[all_labels == 1, 1]   # P(AI) for actual AI samples
ax6.hist(human_probs, bins=40, alpha=0.6, color='#3498db', label='Actual Human', density=True)
ax6.hist(ai_probs,    bins=40, alpha=0.6, color='#e74c3c', label='Actual AI',    density=True)
ax6.axvline(0.5, color='black', linestyle='--', linewidth=1.2, label='Threshold = 0.5')
ax6.set_xlabel("P(AI)")
ax6.set_ylabel("Density")
ax6.set_title("Predicted Probability Distribution")
ax6.legend()
ax6.grid(alpha=0.3)

plt.suptitle("Model Evaluation Report", fontsize=16, fontweight='bold', y=1.01)
plt.savefig("evaluation_report.png", dpi=300, bbox_inches='tight')
plt.show()
print("Evaluation report saved as 'evaluation_report.png'")

In [ ]:
class LMPredictor:
    def __init__(self, model, tokenizer, device, class_names=["Human", "AI"]):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.class_names = class_names

    def predict_proba(self, texts, max_length=512):
        self.model.eval()
        with torch.no_grad():
            encodings = self.tokenizer.batch_encode_plus(
                texts,
                padding="max_length",
                truncation=True,
                return_tensors="pt",
                max_length=max_length
            )

            input_ids = encodings['input_ids'].to(self.device)
            attention_mask = encodings['attention_mask'].to(self.device)

            output = self.model(input_ids, attention_mask)
            logits = output.logits
            probs = torch.softmax(logits, dim=1)
        
        return probs.cpu().numpy()

    def predict_label(self, texts, max_length=512):
        probs = self.predict_proba(texts, max_length=max_length)
        preds_index = probs.argmax(axis=1)
        preds_label = [self.class_names[i] for i in preds_index]
        return preds_label


# Example usage
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)


predictor = LMPredictor(model, tokenizer, device, class_names=["Human", "AI"])


sample_texts = [
    "काठमाडौँमा आज राम्ररी मौसम छ।",
    "यो टेक्स्ट एआईले लेखेको हो।",
    '''काठमाडौँ रविजी नमस्कार सबैभन्दा पहिले त नागरिकता पुनः प्राप्तिको प्रक्रिया पूरा गरी नयाँ नागरिकता लिनुभएकोमा हार्दिक बधाई 
बेथिति विसंगति बेरोजगारी भ्रष्टाचार दण्डहीनताले जकडिएको देशको राजनीतिमा तपाईंको आगमनले खासगरी युवा वर्ग अनि घर र देश धान्न विदेसिएका नेपालीहरूमा 
परम खुसीको सञ्चार भएको थियो त्यसैले तपाईंले त्यसलाई निराशा र दुःखमा परिणत हुन दिन पाउनुहुन्न कांग्रेस एमाले माओवादीलगायतका राजनीतिक दलको बेइमानी 
सत्तालिप्सा र खिचातानी कारण नै तपाईंलाई राजनीतिमा प्रवेश गर्न र छोटो समयमै लोभलाग्दो सफलता हासिल गर्न सहज भएको हो तिनै राजनीतिक दलका नेताले गल्ती
गर्ने र उल्टो बाटोमा हिँड्ने छुट तपाईंलाई छैन विल्कुल छँदै छैन त्यसैले देशको भलाइका लागि तपाईंको लामो राजनीतिक भविष्यको कामनासहित केही सुझाव दिन चाहन्छु 
सबैभन्दा पहिले त तपाईंले नागरिकता पुनः प्राप्तिको प्रक्रिया पूरा नगरेको सर्वोच्च अदालतको फैसला स्वीकार गरिसक्नुभएको छ र नयाँ नागरिकता प्राप्त गरिसकेको हुनाले विगतमा 
जानी वा नजानी आफूबाट गल्ती भएकोमा सावर्जनिक रूपमा माफी माग्नुस् देशको सर्वोच्च अदालतको आदेश स्वीकार गर्नुपर्नेमा म अनागरिक भन्ने तपाईंको अभिव्यक्ति साह्रै 
आलोकाँचो थियो त्यो अभिव्यक्तिमा तपाईंको मपाइँत्त्व झल्किन्थ्यो खासमा नागरिकताको विषय तपाईंको कुनै ठूलो अपराध नभई सामान्य प्राविधिक त्रुटि थियो त्यसैले उक्त त्रुटिप्रति 
क्षमायाचना गर्नुहोस् सार्वजनिक रूपमा थोरै झुक्दा सानो भइँदैन उल्टै उचाइ बढ्छ अर्को कुरा अब तपाईंलाई पुनः सरकारमा जान तपाईंका शुभचिन्तक वा सकाउन खोज्ने दुवैले 
फेरि उचाल्नेछन् त्यसबाट बच्नुहोला सरकारमा जाने गल्ती पुनः नदोहोर्याउनुहोला र तपाईं यो गठबन्धनबाट बाहिरिनुहोला किनभने यो पटकको चुनावमार्फत तपाईंहरूले सरकारमा 
जाने म्यान्डेट पाउनुभएकै थिएन खासमा सरकार नै नबन्ने हकमा तपाईंहरूले कुनै गठबन्धनलाई मत दिनुपर्थ्यो तर बार्गेनिङ गरिगरी सरकारमा जानै हुँदैनथ्यो त्यसमाथि तपाईंले 
आफ्नैबारे अनुसन्धान गरिरहेको निकायसँग सम्बन्धित मन्त्रालयको जिम्मेवारी खोजिखोजी लिएकाले तपाईंका विरोधीको आवाज चर्को मात्र भएको छैन कि तपाईंको नियत गलत थियो 
भन्ने आशंका गर्ने ठाउँ पनि प्रशस्त दिएको छ अर्को तपाईंमाथि लागेको दोहोरो पासपोर्टको विषयको अभियोगबारे प्रहरीले छानबिन थालिसकेको छ सक्नुहुन्छ भने जानअनजानमा के 
भएको हो खुलस्त पार्नुहोस् बोलीले होइन व्यवहारले छानबिनमा सघाउनुहोस् अनि अभियोगको अन्तिम टुंगो नलागेसम्म उपनिर्वाचनमा उम्मेदवार बन्ने वा सार्वजनिक पदमा बस्ने गल्ती 
नगर्नुस् बरू चितवन मा हुने उपनिर्वाचनमा आफ्नो दलका तर्फबाट संसद्मा नेतृत्व लिन सक्ने जनताका आवाज बोल्न सक्ने र सशक्त प्रतिपक्षको भूमिका निर्वाह गर्नसक्ने समाजका 
कुनै लब्धप्रतिष्ठित व्यक्ति वा आफ्नै दलको अन्य कोही नेतालाई उठाएर जिताउनुहोस् र आफू पार्टीको संगठन विस्तारका लागि देशव्यापी अभियानमा लाग्नुहोस् समग्रमा वर्षपछि हुने 
चुनावमा बहुमतको सरकार बनाउनका लागि तयारीमा जुट्नुहोस् आम जनता मतदाता र समर्थकलाई त्यही अनुरूप सम्झाउनुस् यसैगरी नेतृत्वले जे गरे पनि ठिक भन्ने र देवत्वकरण 
गर्ने कार्यबाट बच्नुहोस् सल्लाहकार या सहयोगी नजिककोलाई भन्दा पनि सही सल्लाह दिने कहीँ कतै गलत भयो भने सिधै भन्न सक्ने र समयमै सूचना दिने परिपक्व र राज्य संयन्त्र 
बुझेकोलाई राख्नुहोस् समर्थक र फलोअर्सलाई अरूको अस्तित्व स्विकार्न र जथाभावी नबोल्न भन्नुहोस् त्यसका लागि समर्थकलाई सामाजिक सञ्जाललगायतबाट गालीगलौज गर्ने अशिष्ट 
भाषा बोल्ने धम्की दिनेलगायत गतिविधि नगर्न सार्वजनिक रूपमै आग्रह गर्नुहोस् अन्त्यमा आफ्नो नामको सम्पूर्ण सम्पत्ति सार्वजनिक गर्नुहोस् आयस्रोत खुलाउनुहोस् र राज्यलाई कर 
तिरेको प्रमाण देखाइदिनुहोस् तत्काल यति गर्न सक्नुभयो भने तपाईं देश बनाउनै अमेरिका छाडेर आएकोमा कसैले दुई मत नराख्ला त्यसमाथि समर्थकहरू पनि बढ्ने नै छन् यति 
पनि गर्न सक्नुभएन भने देशको राजनीति सफा गर्न र उच्च राजनीतिक संस्कार सिकाउन फोहोर पोखरीमै हामफालेको भन्ने तपाईंको अभिव्यक्ति हामीले किन पत्याउनुपर्ने याद गर्नुस् 
रविजी तपाईंले शंकाको सुविधा पाउनुहुन्न हिजो तपाईंले प्रश्न धेरै सोध्नुभयो अब तपाईंले पाइलापाइलामा उत्तर दिँदै अघि बढ्नुको विकल्प छैन नत्र त 
उही टुडिखेलमा लगेर नाकको डाँडीमै हान्न मन छ भन्दै टेलिभिजनकै स्कृनमा च्याँठ्ठिनु ठिक हुन्छ होला नि होइन र'''
]

sample_texts = [clean_nepali_text(t) for t in sample_texts]

print("Predicted labels:", predictor.predict_label(sample_texts))
print("Predicted probabilities:", predictor.predict_proba(sample_texts))


In [ ]:
# from lime.lime_text import LimeTextExplainer
# import re
# class NepaliLimeExplainer:
#     def __init__(self):
#         # Use a simple Nepali word tokenizer for LIME
#         self.explainer = LimeTextExplainer(
#             class_names=['negative', 'positive'],
#             split_expression=self.nepali_tokenize_for_lime,
#             bow=False
#         )
    
#     @staticmethod
#     def nepali_tokenize_for_lime(text):
#         # Simple tokenizer that preserves Nepali words
#         return re.findall(r'[\w\u0900-\u097F]+|[^\w\s]', text)
    
#     def explain(self, text, predict_fn, num_features=20):
#         return self.explainer.explain_instance(
#             text,
#             predict_fn,
#             num_features=num_features,
#             num_samples=100
#         )

# # Initialize explainer
# explainer = NepaliLimeExplainer()
# # Text to explain

# # Get explanation
# exp = explainer.explain(nepali_text, predict_proba)

# # Show explanation
# print(f"Explanation for: '{nepali_text}'")
# for feature, weight in exp.as_list():
#     print(f"{feature}: {weight:.4f}")

In [ ]:
# exp.show_in_notebook()

In [ ]:
import torch
import numpy as np
import shap
from scipy.special import softmax

class LMShapWrapper:
    def __init__(self, model, tokenizer, device, class_names=["Human", "AI"]):
        self.model = model.to(device)
        self.tokenizer = tokenizer
        self.device = device
        self.class_names = class_names
        
        # 1. Initialize masker without the 'truncation' argument
        # We handle truncation inside the predict method below
        masker = shap.maskers.Text(self.tokenizer, mask_token="...")
        
        self.explainer = shap.Explainer(
            self.predict, 
            masker=masker,
            output_names=self.class_names
        )

    def predict(self, texts):
        """Standard prediction pipeline for SHAP probabilities"""
        if isinstance(texts, np.ndarray):
            texts = texts.tolist()
        
        texts = [str(t) for t in texts]
        self.model.eval()
        
        with torch.no_grad():
            # Truncation happens here! 
            # This ensures even if SHAP sends a long string, the model only gets 512 tokens.
            encodings = self.tokenizer(
                texts,
                padding=True,
                truncation=True, # Crucial: handle it here
                max_length=512,
                return_tensors="pt"
            ).to(self.device)

            outputs = self.model(**encodings)
            
            # SHAP expects probabilities for its Explainer (not raw logits)
            probs = torch.softmax(outputs.logits, dim=1)
            return probs.cpu().numpy()
    
    def explain(self, text, max_evals=500):
        """
        Runs SHAP on the input text. 
        Note: If text is long, SHAP will still only explain the tokens 
        that the predict() function actually sees (the first 512).
        """
        print(f"Generating explanation")
        
        # shap_values becomes an Explanation object for the single input
        shap_values = self.explainer([text], max_evals=max_evals)[0]
        
        # Calculate final probabilities from SHAP values for display
        final_probs = shap_values.base_values + shap_values.values.sum(axis=0)
        winner_idx = np.argmax(final_probs)
        
        print("\n" + "="*30)
        print(" PREDICTION SUMMARY ")
        print("="*30)
        for i, name in enumerate(self.class_names):
            print(f"{name}: {final_probs[i]:.4f}")
        
        print(f"\nVisualizing: {self.class_names[winner_idx]}")
        # Display the plot
        shap.plots.text(shap_values[:, 1])
        
        return

# --- SETUP AND EXECUTION ---

# 1. Device configuration


In [ ]:

# 2. Initialize the Wrapper (assumes 'model' and 'tokenizer' are already loaded)
shap_wrapper = LMShapWrapper(model, tokenizer, device)

# 3. Run the explanation
# If the text is 1000 tokens, only the first 512 will be processed.
# text_input = "Your long text string here..."
# explanation_object = shap_wrapper.explain(text_input)

def predict_shap(article_text):
    # article_text = clean_nepali_text(article_text)
    shap_wrapper.explain(article_text, max_evals= 1000)

In [ ]:

nepali_text ='''काठमाडौँ रविजी नमस्कार सबैभन्दा पहिले त नागरिकता पुनः प्राप्तिको प्रक्रिया पूरा गरी नयाँ नागरिकता लिनुभएकोमा हार्दिक बधाई 
बेथिति विसंगति बेरोजगारी भ्रष्टाचार दण्डहीनताले जकडिएको देशको राजनीतिमा तपाईंको आगमनले खासगरी युवा वर्ग अनि घर र देश धान्न विदेसिएका नेपालीहरूमा 
परम खुसीको सञ्चार भएको थियो त्यसैले तपाईंले त्यसलाई निराशा र दुःखमा परिणत हुन दिन पाउनुहुन्न कांग्रेस एमाले माओवादीलगायतका राजनीतिक दलको बेइमानी 
सत्तालिप्सा र खिचातानी कारण नै तपाईंलाई राजनीतिमा प्रवेश गर्न र छोटो समयमै लोभलाग्दो सफलता हासिल गर्न सहज भएको हो तिनै राजनीतिक दलका नेताले गल्ती
गर्ने र उल्टो बाटोमा हिँड्ने छुट तपाईंलाई छैन विल्कुल छँदै छैन त्यसैले देशको भलाइका लागि तपाईंको लामो राजनीतिक भविष्यको कामनासहित केही सुझाव दिन चाहन्छु 
सबैभन्दा पहिले त तपाईंले नागरिकता पुनः प्राप्तिको प्रक्रिया पूरा नगरेको सर्वोच्च अदालतको फैसला स्वीकार गरिसक्नुभएको छ र नयाँ नागरिकता प्राप्त गरिसकेको हुनाले विगतमा 
जानी वा नजानी आफूबाट गल्ती भएकोमा सावर्जनिक रूपमा माफी माग्नुस् देशको सर्वोच्च अदालतको आदेश स्वीकार गर्नुपर्नेमा म अनागरिक भन्ने तपाईंको अभिव्यक्ति साह्रै 
आलोकाँचो थियो त्यो अभिव्यक्तिमा तपाईंको मपाइँत्त्व झल्किन्थ्यो खासमा नागरिकताको विषय तपाईंको कुनै ठूलो अपराध नभई सामान्य प्राविधिक त्रुटि थियो त्यसैले उक्त त्रुटिप्रति 
क्षमायाचना गर्नुहोस् सार्वजनिक रूपमा थोरै झुक्दा सानो भइँदैन उल्टै उचाइ बढ्छ अर्को कुरा अब तपाईंलाई पुनः सरकारमा जान तपाईंका शुभचिन्तक वा सकाउन खोज्ने दुवैले 
फेरि उचाल्नेछन् त्यसबाट बच्नुहोला सरकारमा जाने गल्ती पुनः नदोहोर्याउनुहोला र तपाईं यो गठबन्धनबाट बाहिरिनुहोला किनभने यो पटकको चुनावमार्फत तपाईंहरूले सरकारमा 
जाने म्यान्डेट पाउनुभएकै थिएन खासमा सरकार नै नबन्ने हकमा तपाईंहरूले कुनै गठबन्धनलाई मत दिनुपर्थ्यो तर बार्गेनिङ गरिगरी सरकारमा जानै हुँदैनथ्यो त्यसमाथि तपाईंले 
आफ्नैबारे अनुसन्धान गरिरहेको निकायसँग सम्बन्धित मन्त्रालयको जिम्मेवारी खोजिखोजी लिएकाले तपाईंका विरोधीको आवाज चर्को मात्र भएको छैन कि तपाईंको नियत गलत थियो 
भन्ने आशंका गर्ने ठाउँ पनि प्रशस्त दिएको छ अर्को तपाईंमाथि लागेको दोहोरो पासपोर्टको विषयको अभियोगबारे प्रहरीले छानबिन थालिसकेको छ सक्नुहुन्छ भने जानअनजानमा के 
भएको हो खुलस्त पार्नुहोस् बोलीले होइन व्यवहारले छानबिनमा सघाउनुहोस् अनि अभियोगको अन्तिम टुंगो नलागेसम्म उपनिर्वाचनमा उम्मेदवार बन्ने वा सार्वजनिक पदमा बस्ने गल्ती 
नगर्नुस् बरू चितवन मा हुने उपनिर्वाचनमा आफ्नो दलका तर्फबाट संसद्मा नेतृत्व लिन सक्ने जनताका आवाज बोल्न सक्ने र सशक्त प्रतिपक्षको भूमिका निर्वाह गर्नसक्ने समाजका 
कुनै लब्धप्रतिष्ठित व्यक्ति वा आफ्नै दलको अन्य कोही नेतालाई उठाएर जिताउनुहोस् र आफू पार्टीको संगठन विस्तारका लागि देशव्यापी अभियानमा लाग्नुहोस् समग्रमा वर्षपछि हुने 
चुनावमा बहुमतको सरकार बनाउनका लागि तयारीमा जुट्नुहोस् आम जनता मतदाता र समर्थकलाई त्यही अनुरूप सम्झाउनुस् यसैगरी नेतृत्वले जे गरे पनि ठिक भन्ने र देवत्वकरण 
गर्ने कार्यबाट बच्नुहोस् सल्लाहकार या सहयोगी नजिककोलाई भन्दा पनि सही सल्लाह दिने कहीँ कतै गलत भयो भने सिधै भन्न सक्ने र समयमै सूचना दिने परिपक्व र राज्य संयन्त्र 
बुझेकोलाई राख्नुहोस् समर्थक र फलोअर्सलाई अरूको अस्तित्व स्विकार्न र जथाभावी नबोल्न भन्नुहोस् त्यसका लागि समर्थकलाई सामाजिक सञ्जाललगायतबाट गालीगलौज गर्ने अशिष्ट 
भाषा बोल्ने धम्की दिनेलगायत गतिविधि नगर्न सार्वजनिक रूपमै आग्रह गर्नुहोस् अन्त्यमा आफ्नो नामको सम्पूर्ण सम्पत्ति सार्वजनिक गर्नुहोस् आयस्रोत खुलाउनुहोस् र राज्यलाई कर 
तिरेको प्रमाण देखाइदिनुहोस् तत्काल यति गर्न सक्नुभयो भने तपाईं देश बनाउनै अमेरिका छाडेर आएकोमा कसैले दुई मत नराख्ला त्यसमाथि समर्थकहरू पनि बढ्ने नै छन् यति 
पनि गर्न सक्नुभएन भने देशको राजनीति सफा गर्न र उच्च राजनीतिक संस्कार सिकाउन फोहोर पोखरीमै हामफालेको भन्ने तपाईंको अभिव्यक्ति हामीले किन पत्याउनुपर्ने याद गर्नुस् 
रविजी तपाईंले शंकाको सुविधा पाउनुहुन्न हिजो तपाईंले प्रश्न धेरै सोध्नुभयो अब तपाईंले पाइलापाइलामा उत्तर दिँदै अघि बढ्नुको विकल्प छैन नत्र त 
उही टुडिखेलमा लगेर नाकको डाँडीमै हान्न मन छ भन्दै टेलिभिजनकै स्कृनमा च्याँठ्ठिनु ठिक हुन्छ होला नि होइन र'''

predict_shap(nepali_text)


In [ ]:
nepali_text = "नेपाली भाषा हाम्रो दैनिकीको हिस्सा मात्र होइन, हाम्रो भावना र पहिचानसँग गहिरो रूपमा जोडिएको भाषा हो। हामी हाँस्दा, दुःखी हुँदा, माया व्यक्त गर्दा वा आफ्ना कथा सुनाउँदा नेपाली भाषामै बोल्छौँ। नेपाल विभिन्न जातजाति, संस्कृति र परम्पराले भरिएको देश भए पनि नेपाली भाषा सबैलाई एउटै डोरले बाँध्ने माध्यम बनेको छ। यही भाषामार्फत हामी आफ्नो इतिहास सम्झन्छौँ र भविष्यको सपना देख्छौँ।"

predict_shap(nepali_text)

In [ ]:
nepali_text = "नेपाली विकिपिडिया एक स्वतन्त्र विश्वकोश हो। हाल नेपाली विकिपिडियामा २९,३६९ लेखहरू रहेका छन् भने यस विकिपिडियामा ६ जना प्रबन्धकहरू र एक जना प्रशासक रहेका छन्। नेपाली, देवनागरी लिपि प्रयोग गरेर, उपकरणहरूमा टाइप गर्न जटिल ट्रान्सलिटेरेशन एड्स चाहिन्छ। तसर्थ, यसमा कुनै विशेष नेपाली टाइपिङ सफ्टवेयर प्रयोग नगरी नेपालीमा फोनेटिक ल्याटिन वर्णमाला कन्भर्टर छ।"

predict_shap(nepali_text)


In [ ]:
nepali_text = "नेपाली भाषा एक आर्य भाषा हो जुन दक्षिण एसियाको हिमालय क्षेत्रमा बोलिन्छ। यो नेपालको आधिकारिक, र सबैभन्दा व्यापक रूपमा बोलिने भाषा हो, जहाँ यसले सम्पर्क भाषाको रूपमा पनि काम गर्दछ। भारतीय राज्य सिक्किम र पश्चिम बङ्गालको गोर्खाल्यान्ड क्षेत्रीय प्रशासनमा नेपाली भाषाको आधिकारिक हैसियत छ र अरुणाचल प्रदेश, असम, हिमाचल प्रदेश, मणिपुर, मेघालय, मिजोरम र उत्तराखण्ड राज्यहरूमा पनि नेपाली बोल्नेहरूको उल्लेखनीय सङ्ख्या छ। नेपाली भाषा भुटानको लगभग एक चौथाई जनसङ्ख्या द्वारा बोलिन्छ। म्यानमारमा यो भाषा बर्मेली नेपालीले बोल्छन्। मध्यपूर्व, ब्रुनाई, अस्ट्रेलिया र विश्वभरका विदेशमा रहेका नेपालीले पनि यो भाषा प्रयोग गर्छन्। नेपाली लगभग १.६ करोड मातृभाषीहरू र अर्को ९ लाख मानिसले दोस्रो भाषाको रूपमा बोल्छन्।"

predict_shap(nepali_text)


In [ ]:
nepali_text = "काठमाडौँ — पोखराको बाटुलेचौरस्थित लिची बगानको जग्गा हिनामिना प्रपञ्चमा घूस लिएको अभियोग लागेका पूर्वमन्त्री राजकुमार गुप्तालाई पुर्पक्षका लागि थुनामा पठाउन विशेष अदालतले आदेश गरेको छ । न्यायाधीशहरू सुदर्शनदेव भट्ट, डिल्लीरत्न श्रेष्ठ र विदुर कोइरालाको इजलासले बुधबार गुप्तालाई थुना पुर्जी दिई कारागार पठाउन आदेश गरेको हो ।"

predict_shap(nepali_text)


In [ ]:
nepali_text = "विशेष अदालतमा बुधबार बिहान ११ बजे हाजिर भएका गुप्ताको दिनभर बयान भएको थियो । बयानमा उनले आफूमाथि लागेको आरोप झूटो भएको र कसैसँग घूस नलिएको दाबी गरेका थिए । तर अदालतले तत्काल प्राप्त प्रमाणका आधारमा निर्दोष मान्न नसकिने भन्दै पुर्पक्षका लागि थुनामा पठाउने आदेश गरेको छ ।"

predict_shap(nepali_text)


In [ ]:
nepali_text = " पोखराको बाटुलेचौरस्थित लिची बगानको जग्गा हडप्ने प्रपञ्चमा दुई पूर्वमन्त्री राजकुमार गुप्ता र रञ्जिता श्रेष्ठविरुद्ध २३ असोजमा अख्तियार दुरुपयोग अनुसन्धान आयोगले भ्रष्टाचार मुद्दा दायर गरेको थियो । बिचौलिया सुजनकुमार तामाङका साथै खेमबहादुर पुन, यामकुमारी गुरुङ र तुल्सीराम बुढामगरलाई पनि प्रतिवादी बनाइएको थियो । एमाले नेतासमेत रहेका पूर्वसामान्य प्रशासनमन्त्री गुप्ता र नागरिक उन्मुक्ति पार्टीकी अध्यक्ष एवं पूर्वभूमि व्यवस्था, सहकारी तथा गरिबी निवारणमन्त्री श्रेष्ठविरुद्ध ७८ लाख रुपैयाँका दरले बिगो मागदाबी गरिएको थियो । पूर्वमन्त्री श्रेष्ठ भने अदालतमा हाजिर भएकी छैनन् ।"

predict_shap(nepali_text)
